# Synthetic Data Generation from Random DAGs

This notebook starts a NOTEARS-style synthetic data pipeline by generating random directed acyclic graphs (DAGs). The core helper below uses an Erdos-Renyi edge sampling process over a random topological ordering, which guarantees acyclicity and avoids disconnected nodes.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import combinations
from math import erfc, sqrt
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import networkx as nx


## Random Erdos-Renyi DAG generator

For a DAG with `n_nodes`, a random permutation defines a hidden topological order. A random spanning chain is added first so every node has at least one incident edge. Remaining edges are sampled only from earlier nodes to later nodes in that order, so cycles cannot be introduced.

You can control density in either of two ways:

- `edge_prob`: direct Bernoulli probability for each possible ordered edge.
- `expected_degree`: expected number of incident edges per node, converted to `edge_prob = expected_degree / (n_nodes - 1)`.

In [ ]:
@dataclass(frozen=True)
class RandomDag:
    adjacency: np.ndarray
    weights: np.ndarray
    topological_order: np.ndarray


def generate_er_dag(
    n_nodes: int = 15,
    edge_prob: Optional[float] = None,
    expected_degree: float = 2.0,
    weight_range: Tuple[float, float] = (0.5, 2.0),
    seed: Optional[int] = None,
) -> RandomDag:
    """Generate a connected random weighted DAG with an Erdos-Renyi edge process.

    Parameters
    ----------
    n_nodes:
        Number of variables/nodes. NOTEARS-style examples often use values
        around 10 to 20; the default here is 15.
    edge_prob:
        Probability of each possible DAG-compatible directed edge. If omitted,
        it is derived from ``expected_degree``.
    expected_degree:
        Approximate expected number of neighbors per node. Used only when
        ``edge_prob`` is not supplied.
    weight_range:
        Absolute range for sampled edge weights. Signs are assigned randomly.
    seed:
        Optional random seed for reproducibility.

    Returns
    -------
    RandomDag
        ``adjacency`` is binary with shape ``(n_nodes, n_nodes)`` and
        ``weights`` contains signed edge weights in the same orientation.
        Every node has at least one incident edge.
    """
    if n_nodes < 2:
        raise ValueError("n_nodes must be at least 2")

    if edge_prob is None:
        edge_prob = expected_degree / (n_nodes - 1)

    if not 0 <= edge_prob <= 1:
        raise ValueError("edge_prob must be between 0 and 1")

    low, high = weight_range
    if low <= 0 or high <= low:
        raise ValueError("weight_range must satisfy 0 < low < high")

    rng = np.random.default_rng(seed)
    order = rng.permutation(n_nodes)

    adjacency = np.zeros((n_nodes, n_nodes), dtype=int)

    # Add a random spanning chain along the topological order. This keeps the
    # graph acyclic while ensuring no node is isolated.
    for source, target in zip(order[:-1], order[1:]):
        adjacency[source, target] = 1

    for source_pos in range(n_nodes - 1):
        source = order[source_pos]
        candidate_targets = order[source_pos + 1 :]
        edge_mask = rng.random(len(candidate_targets)) < edge_prob
        adjacency[source, candidate_targets[edge_mask]] = 1

    magnitudes = rng.uniform(low, high, size=(n_nodes, n_nodes))
    signs = rng.choice([-1.0, 1.0], size=(n_nodes, n_nodes))
    weights = adjacency * magnitudes * signs

    return RandomDag(adjacency=adjacency, weights=weights, topological_order=order)


def is_acyclic(adjacency: np.ndarray) -> bool:
    """Check acyclicity with Kahn's algorithm."""
    adjacency = np.asarray(adjacency)
    in_degree = adjacency.sum(axis=0).astype(int)
    queue = [node for node, degree in enumerate(in_degree) if degree == 0]
    visited = 0

    while queue:
        node = queue.pop()
        visited += 1
        for child in np.flatnonzero(adjacency[node]):
            in_degree[child] -= 1
            if in_degree[child] == 0:
                queue.append(child)

    return visited == adjacency.shape[0]


def has_no_disconnected_nodes(adjacency: np.ndarray) -> bool:
    """Return True when every node has at least one incoming or outgoing edge."""
    adjacency = np.asarray(adjacency)
    incident_edges = adjacency.sum(axis=0) + adjacency.sum(axis=1)
    return bool(np.all(incident_edges > 0))

## Example DAG with 5 nodes


In [ ]:
dag = generate_er_dag(n_nodes=7, expected_degree=2.0, seed=43)

print(f"Nodes: {dag.adjacency.shape[0]}")
print(f"Edges: {dag.adjacency.sum()}")
print(f"Acyclic: {is_acyclic(dag.adjacency)}")
print(f"No disconnected nodes: {has_no_disconnected_nodes(dag.adjacency)}")
print(f"Topological order: {dag.topological_order.tolist()}")

dag.adjacency

In [ ]:
def plot_dag(adjacency: np.ndarray, topological_order: Optional[np.ndarray] = None) -> None:
    """Plot a DAG when networkx is available, otherwise show its adjacency matrix."""
    adjacency = np.asarray(adjacency)

    if nx is None:
        plt.figure(figsize=(5, 5))
        plt.imshow(adjacency, cmap="Greys", interpolation="nearest")
        node_labels = np.arange(adjacency.shape[0])
        plt.xticks(node_labels, node_labels)
        plt.yticks(node_labels, node_labels)
        plt.xlabel("Target node")
        plt.ylabel("Source node")
        plt.title("DAG adjacency matrix")
        plt.colorbar(label="Edge present")
        plt.show()
        return

    graph = nx.from_numpy_array(adjacency, create_using=nx.DiGraph)
    labels = {node: str(node) for node in graph.nodes}

    if topological_order is None:
        topological_order = np.array(list(nx.topological_sort(graph)))

    pos = nx.circular_layout(graph)

    plt.figure(figsize=(5, 5))
    nx.draw_networkx_nodes(graph, pos, node_size=850, node_color="#e8f3ff", edgecolors="#1f2937")
    nx.draw_networkx_labels(graph, pos, labels=labels, font_size=10)
    nx.draw_networkx_edges(
        graph, pos, arrows=True, arrowstyle="-|>",
        arrowsize=18, width=2.0, edge_color="#b42318"
    )
    plt.title("Random Erdos-Renyi DAG")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


plot_dag(dag.adjacency, dag.topological_order)

## Synthetic data from the weighted DAG

The generated graph gives a binary adjacency matrix `B`. The generator also assigns edge weights independently from `Unif([-2, -0.5] union [0.5, 2])`, represented by `dag.weights`.

For Gaussian synthetic data, each observation is sampled from the linear SEM:

`x = W.T @ x + z`, where `z ~ N(0, I)`.

Because `W` is a DAG, this can be evaluated in topological order instead of solving a simultaneous cyclic system.

In [ ]:
def sample_gaussian_linear_sem(
    weights: np.ndarray,
    topological_order: np.ndarray,
    n_samples: int = 1000,
    noise_scale: float = 0.1,
    seed: Optional[int] = None,
) -> np.ndarray:
    """Sample data from x = W.T @ x + z with z ~ N(0, noise_scale^2 I).

    ``weights[parent, child]`` stores the structural coefficient for
    parent -> child. Variables are evaluated in topological order.
    """
    weights = np.asarray(weights, dtype=float)
    topological_order = np.asarray(topological_order, dtype=int)

    if weights.ndim != 2 or weights.shape[0] != weights.shape[1]:
        raise ValueError("weights must be a square matrix")

    n_nodes = weights.shape[0]
    if sorted(topological_order.tolist()) != list(range(n_nodes)):
        raise ValueError("topological_order must contain each node exactly once")

    if n_samples <= 0:
        raise ValueError("n_samples must be positive")

    if noise_scale <= 0:
        raise ValueError("noise_scale must be positive")

    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=0.0, scale=noise_scale, size=(n_samples, n_nodes))
    samples = np.zeros((n_samples, n_nodes), dtype=float)

    for child in topological_order:
        parents = np.flatnonzero(weights[:, child])
        if len(parents) == 0:
            samples[:, child] = noise[:, child]
        else:
            samples[:, child] = samples[:, parents] @ weights[parents, child] + noise[:, child]

    return samples

In [ ]:
X = sample_gaussian_linear_sem(
    weights=dag.weights,
    topological_order=dag.topological_order,
    n_samples=10000,
    seed=123,
)

print(f"Synthetic data shape: {X.shape}")
print(f"Feature means, first 5: {np.round(X.mean(axis=0)[:5], 3)}")
print(f"Feature stds, first 5: {np.round(X.std(axis=0)[:5], 3)}")

X[:5]

## PC-stable causal discovery

The PC-stable procedure below estimates a graph from the synthetic Gaussian data using partial-correlation conditional independence tests. It first learns a stable skeleton and then orients unshielded colliders where possible. Edges that cannot be oriented are shown as undirected.

In [ ]:
@dataclass(frozen=True)
class PcStableResult:
    skeleton: np.ndarray
    directed: np.ndarray
    sep_sets: dict[tuple[int, int], set[int]]


def _normal_two_sided_p_value(z_score: float) -> float:
    """Two-sided p-value for a standard normal z-score."""
    return erfc(abs(z_score) / sqrt(2.0))


def partial_correlation_p_value(
    data: np.ndarray,
    x: int,
    y: int,
    conditioning_set: tuple[int, ...] = (),
) -> tuple[float, float]:
    """Return partial correlation and Fisher-z p-value for X_x independent X_y | S."""
    data = np.asarray(data, dtype=float)
    n_samples = data.shape[0]
    variables = [x, y, *conditioning_set]

    if n_samples <= len(conditioning_set) + 3:
        return 0.0, 0.0

    corr = np.corrcoef(data[:, variables], rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)

    if len(conditioning_set) == 0:
        r = corr[0, 1]
    else:
        precision = np.linalg.pinv(corr)
        denom = np.sqrt(max(precision[0, 0] * precision[1, 1], 1e-12))
        r = -precision[0, 1] / denom

    r = float(np.clip(r, -0.999999, 0.999999))
    fisher_z = 0.5 * np.log((1.0 + r) / (1.0 - r))
    test_statistic = np.sqrt(n_samples - len(conditioning_set) - 3) * fisher_z
    p_value = _normal_two_sided_p_value(test_statistic)
    return r, p_value


def pc_stable(
    data: np.ndarray,
    alpha: float = 0.01,
    max_conditioning_set_size: Optional[int] = None,
) -> PcStableResult:
    """Estimate a CPDAG-like graph using the PC-stable skeleton phase."""
    data = np.asarray(data, dtype=float)
    n_nodes = data.shape[1]
    skeleton = np.ones((n_nodes, n_nodes), dtype=bool)
    np.fill_diagonal(skeleton, False)
    sep_sets: dict[tuple[int, int], set[int]] = {}

    if max_conditioning_set_size is None:
        max_conditioning_set_size = n_nodes - 2

    depth = 0
    while depth <= max_conditioning_set_size:
        stable_snapshot = skeleton.copy()
        any_tested = False

        for x, y in combinations(range(n_nodes), 2):
            if not stable_snapshot[x, y]:
                continue

            neighbors = [node for node in np.flatnonzero(stable_snapshot[x]) if node != y]
            if len(neighbors) < depth:
                continue

            any_tested = True
            for conditioning_set in combinations(neighbors, depth):
                _, p_value = partial_correlation_p_value(data, x, y, conditioning_set)
                if p_value > alpha:
                    skeleton[x, y] = False
                    skeleton[y, x] = False
                    sep_sets[(x, y)] = set(conditioning_set)
                    sep_sets[(y, x)] = set(conditioning_set)
                    break

        if not any_tested:
            break
        depth += 1

    directed = np.zeros((n_nodes, n_nodes), dtype=bool)
    for x, z in combinations(range(n_nodes), 2):
        if skeleton[x, z]:
            continue

        common_neighbors = np.flatnonzero(skeleton[x] & skeleton[z])
        for y in common_neighbors:
            if y not in sep_sets.get((x, z), set()):
                directed[x, y] = True
                directed[z, y] = True

    return PcStableResult(
        skeleton=skeleton.astype(int),
        directed=directed.astype(int),
        sep_sets=sep_sets,
    )

In [ ]:
def plot_pc_graph(result: PcStableResult) -> None:
    """Plot directed collider edges and remaining undirected skeleton edges."""
    graph = nx.DiGraph()
    n_nodes = result.skeleton.shape[0]
    graph.add_nodes_from(range(n_nodes))

    directed_edges = [
        (source, target)
        for source in range(n_nodes)
        for target in range(n_nodes)
        if result.directed[source, target]
    ]
    undirected_edges = [
        (source, target)
        for source, target in combinations(range(n_nodes), 2)
        if result.skeleton[source, target]
        and not result.directed[source, target]
        and not result.directed[target, source]
    ]

    graph.add_edges_from(directed_edges + undirected_edges)
    pos = nx.circular_layout(graph)

    plt.figure(figsize=(5, 5))
    nx.draw_networkx_nodes(graph, pos, node_size=850, node_color="#e8f3ff", edgecolors="#1f2937")
    labels = {node: str(node) for node in graph.nodes}
    nx.draw_networkx_labels(graph, pos, labels=labels, font_size=10)
    nx.draw_networkx_edges(
        graph, pos, edgelist=directed_edges, arrows=True, arrowstyle="-|>",
        arrowsize=18, width=2.0, edge_color="#b42318"
    )
    nx.draw_networkx_edges(
        graph, pos, edgelist=undirected_edges, arrows=False,
        width=2.0, style="dashed", edge_color="#344054"
    )
    plt.title("PC-stable estimated graph")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


pc_result = pc_stable(X, alpha=0.01)
print(f"Estimated skeleton edges: {pc_result.skeleton.sum() // 2}")
print(f"Oriented edges: {pc_result.directed.sum()}")
plot_pc_graph(pc_result)